In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
spark = SparkSession.builder \
    .appName("CoursesAnalysis") \
    .getOrCreate()

print(spark.version)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/24 17:44:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
df = spark.read.csv(
    "hdfs://master:9000/user/hadoop/data/courses.csv",
    header=True,
    inferSchema=True
)
# RDD part 
rdd = df.rdd
# Dataset part 
df.createOrReplaceTempView("courses")


In [5]:
df.printSchema()

root
 |-- section: string (nullable = true)
 |-- class: string (nullable = true)
 |-- time: integer (nullable = true)
 |-- teacher: string (nullable = true)
 |-- university: string (nullable = true)
 |-- year: integer (nullable = true)



In [6]:
df.show(5)


+----------------+--------------------+----+-----------+----------+----+
|         section|               class|time|    teacher|university|year|
+----------------+--------------------+----+-----------+----------+----+
|Computer Science|Introduction to P...|  45| M. BENAMOR|     FSEGS|2022|
|     Mathematics|          Calculus I|  60|   A. TLILI|     FSEGS|2025|
|       Economics|      Microeconomics|  50|K. HAMROUNI|       FSS|2022|
|     Engineering|      Circuit Theory|  50|R. MASMOUDI|      ENIS|2025|
|Computer Science|    Database Systems|  55|S. GHOZLANE|     ISIMS|2022|
+----------------+--------------------+----+-----------+----------+----+
only showing top 5 rows



###  Q1 : Compter les cours par section

In [7]:
# RDD 
rdd.map(lambda row: (row["section"], 1)) \
   .reduceByKey(lambda a, b: a + b) \
   .sortBy(lambda x: x[1], ascending=False) \
   .collect()

[('Computer Science', 36),
 ('Engineering', 19),
 ('Mathematics', 18),
 ('Economics', 18),
 ('Business', 17)]

In [8]:
# Dataframe 
df.groupBy("section") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

+----------------+-----+
|         section|count|
+----------------+-----+
|Computer Science|   36|
|     Engineering|   19|
|     Mathematics|   18|
|       Economics|   18|
|        Business|   17|
+----------------+-----+



In [9]:
# Dataset 
spark.sql("""
    SELECT section, COUNT(*) as count
    FROM courses
    GROUP BY section
    ORDER BY count DESC
""").show()


+----------------+-----+
|         section|count|
+----------------+-----+
|Computer Science|   36|
|     Engineering|   19|
|     Mathematics|   18|
|       Economics|   18|
|        Business|   17|
+----------------+-----+



### Q2: Moyenne du temps par section

In [10]:
# RDD
rdd.map(lambda row: (row["section"], (float(row["time"]), 1))) \
   .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])) \
   .map(lambda x: (x[0], round(x[1][0]/x[1][1], 2))) \
   .sortBy(lambda x: x[1], ascending=False) \
   .collect()

[('Mathematics', 55.89),
 ('Engineering', 53.79),
 ('Economics', 50.83),
 ('Computer Science', 50.03),
 ('Business', 48.24)]

In [11]:
# Dataframe
df.groupBy("section").agg(F.avg("time").alias("moy")).orderBy("moy", ascending=False).show()


+----------------+------------------+
|         section|               moy|
+----------------+------------------+
|     Mathematics|55.888888888888886|
|     Engineering| 53.78947368421053|
|       Economics|50.833333333333336|
|Computer Science| 50.02777777777778|
|        Business| 48.23529411764706|
+----------------+------------------+



In [12]:
# Dataset 
spark.sql("""
    SELECT section, AVG(time) as moy
    FROM courses
    GROUP BY section
    ORDER BY moy DESC
""").show()

+----------------+------------------+
|         section|               moy|
+----------------+------------------+
|     Mathematics|55.888888888888886|
|     Engineering| 53.78947368421053|
|       Economics|50.833333333333336|
|Computer Science| 50.02777777777778|
|        Business| 48.23529411764706|
+----------------+------------------+



### Q3: Compter les cours par université

In [13]:
# RDD
rdd.map(lambda row: (row["university"], 1)) \
   .reduceByKey(lambda a, b: a + b) \
   .sortBy(lambda x: x[1], ascending=False) \
   .collect()

[('FSS', 35), ('FSEGS', 31), ('ENIS', 19), ('ISIMS', 12), ("ENET'Com", 11)]

In [14]:
# Dataframe
df.groupBy("university").count().orderBy("count", ascending=False).show()

+----------+-----+
|university|count|
+----------+-----+
|       FSS|   35|
|     FSEGS|   31|
|      ENIS|   19|
|     ISIMS|   12|
|  ENET'Com|   11|
+----------+-----+



In [15]:
# Dataset
spark.sql("""
    SELECT university, COUNT(*) as count
    FROM courses
    GROUP BY university
    ORDER BY count DESC
""").show()

+----------+-----+
|university|count|
+----------+-----+
|       FSS|   35|
|     FSEGS|   31|
|      ENIS|   19|
|     ISIMS|   12|
|  ENET'Com|   11|
+----------+-----+



### Q4: Cours le plus long par section

In [16]:
# RDD 
rdd.map(lambda row: (row["section"], (row["class"], float(row["time"])))) \
   .reduceByKey(lambda a, b: a if a[1] >= b[1] else b) \
   .collect()

[('Computer Science', ('Algorithms', 66.0)),
 ('Mathematics', ('Algebra', 61.0)),
 ('Economics', ('International Trade', 55.0)),
 ('Engineering', ('Structural Analysis', 62.0)),
 ('Business', ('Business Analytics', 65.0))]

In [17]:
# Dataframe
window = Window.partitionBy("section").orderBy(F.col("time").desc())
df.withColumn("rank", F.rank().over(window)) \
  .filter(F.col("rank") == 1) \
  .select("section", "class", "time") \
  .show()

+----------------+--------------------+----+
|         section|               class|time|
+----------------+--------------------+----+
|        Business|  Business Analytics|  65|
|Computer Science|          Algorithms|  66|
|       Economics| International Trade|  55|
|       Economics|Behavioral Economics|  55|
|       Economics|Environmental Eco...|  55|
|       Economics|     Urban Economics|  55|
|       Economics| Financial Economics|  55|
|       Economics|Experimental Econ...|  55|
|       Economics|         Game Theory|  55|
|     Engineering| Structural Analysis|  62|
|     Mathematics|             Algebra|  61|
+----------------+--------------------+----+



In [18]:
# Dataset 
spark.sql("""
    SELECT section, class, time
    FROM (
        SELECT section, class, time,
        RANK() OVER (PARTITION BY section ORDER BY time DESC) as rank
        FROM courses
    )
    WHERE rank = 1
""").show()

+----------------+--------------------+----+
|         section|               class|time|
+----------------+--------------------+----+
|        Business|  Business Analytics|  65|
|Computer Science|          Algorithms|  66|
|       Economics| International Trade|  55|
|       Economics|Behavioral Economics|  55|
|       Economics|Environmental Eco...|  55|
|       Economics|     Urban Economics|  55|
|       Economics| Financial Economics|  55|
|       Economics|Experimental Econ...|  55|
|       Economics|         Game Theory|  55|
|     Engineering| Structural Analysis|  62|
|     Mathematics|             Algebra|  61|
+----------------+--------------------+----+



### Q5 : Compter les cours par année

In [19]:
# RDD
rdd.map(lambda row: (row["year"], 1)) \
   .reduceByKey(lambda a, b: a + b) \
   .sortBy(lambda x: x[0]) \
   .collect()

[(2022, 37), (2023, 22), (2024, 31), (2025, 18)]

In [20]:
# Dataframe
df.groupBy("year").count().orderBy("year").show()

+----+-----+
|year|count|
+----+-----+
|2022|   37|
|2023|   22|
|2024|   31|
|2025|   18|
+----+-----+



In [21]:
# Dataset
spark.sql("""
    SELECT year, COUNT(*) as count
    FROM courses
    GROUP BY year
    ORDER BY year
""").show()

+----+-----+
|year|count|
+----+-----+
|2022|   37|
|2023|   22|
|2024|   31|
|2025|   18|
+----+-----+



### Q6 : Professeurs avec le plus de cours

In [22]:
# RDD
rdd.map(lambda row: (row["teacher"], 1)) \
   .reduceByKey(lambda a, b: a + b) \
   .sortBy(lambda x: x[1], ascending=False) \
   .take(10)

[('F. BOUKHRIS', 7),
 ('M. BOUAZIZ', 6),
 ('M. ZAIDI', 5),
 ('K. BOUGHLA', 3),
 ('K. BOUAOUN', 2),
 ('M. BENAMOR', 1),
 ('A. TLILI', 1),
 ('K. HAMROUNI', 1),
 ('R. MASMOUDI', 1),
 ('S. GHOZLANE', 1)]

In [23]:
# Dataframe 
df.groupBy("teacher").count().orderBy("count", ascending=False).show()

+-------------+-----+
|      teacher|count|
+-------------+-----+
|  F. BOUKHRIS|    7|
|   M. BOUAZIZ|    6|
|     M. ZAIDI|    5|
|   K. BOUGHLA|    3|
|   K. BOUAOUN|    2|
|   T. MAHJOUB|    1|
|   N. BOUAOUN|    1|
| Y. BOUGHANMI|    1|
|     T. TLILI|    1|
|  S. BENYAHIA|    1|
|  N. BOUKTHIR|    1|
|R. BENYOUSSEF|    1|
| B. BENHASSEN|    1|
|  R. BOUGHNIM|    1|
|  Q. MASSOUDI|    1|
|   I. MARZOUK|    1|
|     D. JEMLI|    1|
| Z. FERCHICHI|    1|
|    F. JEBALI|    1|
|     Y. HAMDI|    1|
+-------------+-----+
only showing top 20 rows



In [24]:
# Dataset 
spark.sql("""
    SELECT teacher, COUNT(*) as count
    FROM courses
    GROUP BY teacher
    ORDER BY count DESC
""").show()

+-------------+-----+
|      teacher|count|
+-------------+-----+
|  F. BOUKHRIS|    7|
|   M. BOUAZIZ|    6|
|     M. ZAIDI|    5|
|   K. BOUGHLA|    3|
|   K. BOUAOUN|    2|
|   T. MAHJOUB|    1|
|   N. BOUAOUN|    1|
| Y. BOUGHANMI|    1|
|     T. TLILI|    1|
|  S. BENYAHIA|    1|
|  N. BOUKTHIR|    1|
|R. BENYOUSSEF|    1|
| B. BENHASSEN|    1|
|  R. BOUGHNIM|    1|
|  Q. MASSOUDI|    1|
|   I. MARZOUK|    1|
|     D. JEMLI|    1|
| Z. FERCHICHI|    1|
|    F. JEBALI|    1|
|     Y. HAMDI|    1|
+-------------+-----+
only showing top 20 rows



### Q7 : Université avec la plus longue durée moyenne

In [25]:
# RDD 
rdd.map(lambda row: (row["university"], (float(row["time"]), 1))) \
   .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])) \
   .map(lambda x: (x[0], round(x[1][0]/x[1][1], 2))) \
   .sortBy(lambda x: x[1], ascending=False) \
   .first()

('ENIS', 53.79)

In [26]:
# Dataframe
df.groupBy("university").agg(F.avg("time").alias("moy")).orderBy("moy", ascending=False).limit(1).show()


+----------+-----------------+
|university|              moy|
+----------+-----------------+
|      ENIS|53.78947368421053|
+----------+-----------------+



In [27]:
# Dataset 
spark.sql("""
    SELECT university, AVG(time) as moy
    FROM courses
    GROUP BY university
    ORDER BY moy DESC
    LIMIT 1
""").show()


+----------+-----------------+
|university|              moy|
+----------+-----------------+
|      ENIS|53.78947368421053|
+----------+-----------------+



### Q8 : Durée la plus fréquente

In [28]:
# RDD
rdd.map(lambda row: (row["time"], 1)) \
   .reduceByKey(lambda a, b: a + b) \
   .sortBy(lambda x: x[1], ascending=False) \
   .first()


(50, 31)

In [29]:
# Dataframe
df.groupBy("time").count().orderBy("count", ascending=False).limit(1).show()


+----+-----+
|time|count|
+----+-----+
|  50|   31|
+----+-----+



In [30]:
# Dataset 
spark.sql("""
    SELECT time, COUNT(*) as count
    FROM courses
    GROUP BY time
    ORDER BY count DESC
    LIMIT 1
""").show()


+----+-----+
|time|count|
+----+-----+
|  50|   31|
+----+-----+



### Q9 : Cours au dessus de la durée moyenne

In [31]:
# RDD

avg = rdd.map(lambda row: float(row["time"])).sum() / rdd.count()
print(f"Moyenne : {avg:.2f}")

rdd.filter(lambda row: float(row["time"]) > avg) \
   .map(lambda row: (row["class"], row["time"], row["section"])) \
   .sortBy(lambda x: x[1], ascending=False) \
   .collect()

Moyenne : 51.52


[('Algorithms', 66, 'Computer Science'),
 ('Business Analytics', 65, 'Business'),
 ('Structural Analysis', 62, 'Engineering'),
 ('Algebra', 61, 'Mathematics'),
 ('Calculus I', 60, 'Mathematics'),
 ('Thermodynamics', 60, 'Engineering'),
 ('Data Structures', 60, 'Computer Science'),
 ('Operating Systems', 60, 'Computer Science'),
 ('Statistics', 60, 'Mathematics'),
 ('Control Systems', 60, 'Engineering'),
 ('Robotics', 60, 'Engineering'),
 ('Numerical Analysis', 60, 'Mathematics'),
 ('Biomedical Engineering', 60, 'Engineering'),
 ('Complex Analysis', 60, 'Mathematics'),
 ('Quantum Computing', 60, 'Computer Science'),
 ('Geometry', 60, 'Mathematics'),
 ('Computational Theory', 60, 'Computer Science'),
 ('Set Theory', 60, 'Mathematics'),
 ('Graph Theory', 60, 'Mathematics'),
 ('Functional Programming', 60, 'Computer Science'),
 ('Logic', 60, 'Mathematics'),
 ('Computer Architecture', 60, 'Computer Science'),
 ('Database Systems', 55, 'Computer Science'),
 ('Linear Algebra', 55, 'Mathematic

In [32]:
# Dataframe
avg = df.agg(F.avg("time")).collect()[0][0]
print(f"Moyenne : {avg:.2f}")

df.filter(F.col("time") > avg) \
  .select("class", "time", "section") \
  .orderBy("time", ascending=False) \
  .show()

Moyenne : 51.52
+--------------------+----+----------------+
|               class|time|         section|
+--------------------+----+----------------+
|          Algorithms|  66|Computer Science|
|  Business Analytics|  65|        Business|
| Structural Analysis|  62|     Engineering|
|             Algebra|  61|     Mathematics|
|          Calculus I|  60|     Mathematics|
|      Thermodynamics|  60|     Engineering|
|     Data Structures|  60|Computer Science|
|   Operating Systems|  60|Computer Science|
|          Statistics|  60|     Mathematics|
|     Control Systems|  60|     Engineering|
|            Robotics|  60|     Engineering|
|  Numerical Analysis|  60|     Mathematics|
|Biomedical Engine...|  60|     Engineering|
|    Complex Analysis|  60|     Mathematics|
|   Quantum Computing|  60|Computer Science|
|            Geometry|  60|     Mathematics|
|Computational Theory|  60|Computer Science|
|          Set Theory|  60|     Mathematics|
|        Graph Theory|  60|     Mathema

In [33]:
# Dataset 
spark.sql("""
    SELECT class, time, section
    FROM courses
    WHERE time > (SELECT AVG(time) FROM courses)
    ORDER BY time DESC
""").show()

+--------------------+----+----------------+
|               class|time|         section|
+--------------------+----+----------------+
|          Algorithms|  66|Computer Science|
|  Business Analytics|  65|        Business|
| Structural Analysis|  62|     Engineering|
|             Algebra|  61|     Mathematics|
|          Calculus I|  60|     Mathematics|
|      Thermodynamics|  60|     Engineering|
|     Data Structures|  60|Computer Science|
|   Operating Systems|  60|Computer Science|
|          Statistics|  60|     Mathematics|
|     Control Systems|  60|     Engineering|
|            Robotics|  60|     Engineering|
|  Numerical Analysis|  60|     Mathematics|
|Biomedical Engine...|  60|     Engineering|
|    Complex Analysis|  60|     Mathematics|
|   Quantum Computing|  60|Computer Science|
|            Geometry|  60|     Mathematics|
|Computational Theory|  60|Computer Science|
|          Set Theory|  60|     Mathematics|
|        Graph Theory|  60|     Mathematics|
|Functiona

In [34]:
spark.stop()